# Grid UCI BNN — Skeleton Diagnostics

Loads raw skeleton files from `uci_bnn_grid_skeleton.py` (`results/grid/uci_bnn_skeleton/...`)
and inspects sampler mechanics directly: the Poisson rate $\lambda(t)$ against its grid
bound, event-type composition, and bound tightness.

(Goan et al.'s TF/TFP samplers were investigated here in an earlier version of this notebook
and removed: their hull is built fresh, from scratch, inside each bounce attempt and never
persisted anywhere — the skeleton files only contain the *outcome* of that process
(`acceptance_ratio` at the accepted point), not the hull itself or the rejected proposals that
built it, so it cannot be reconstructed from saved data. `acceptance_ratio` alone is still a
valid, useful number (~90% of Boston TF Boomerang events exceed their own bound), just not one
this notebook's per-segment rate-vs-bound figures can meaningfully visualize.)


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import Tensor

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

DATASET, SPLIT_ID = "boston", 0
SKELETON_DIR = Path("results/grid/uci_bnn_skeleton")

LABELS = {"grid_zigzag": "Grid ZigZag", "grid_sticky_zigzag": "Grid Sticky ZigZag",
          "grid_boomerang": "Grid Boomerang", "grid_sticky_boomerang": "Grid Sticky Boomerang"}
COLORS = {"grid_zigzag": "#1F77B4", "grid_sticky_zigzag": "#1F77B4",
          "grid_boomerang": "#FA5C00", "grid_sticky_boomerang": "#FA5C00"}
LINESTYLES = {"grid_zigzag": "--", "grid_sticky_zigzag": "-",
              "grid_boomerang": "--", "grid_sticky_boomerang": "-"}

skel_dir = SKELETON_DIR / DATASET / f"split_{SPLIT_ID:02d}"
# Only grid_*_skeleton.pt files -- Goan's tf_bps/tf_boomerang skeletons are
# not used by this notebook (see markdown above).
skel_files = sorted(p for p in skel_dir.glob("*_skeleton.pt")
                     if p.stem.removesuffix("_skeleton") in LABELS)
assert skel_files, f"No grid skeleton .pt files found in {skel_dir}."

skeletons: dict[str, dict] = {}
for p in skel_files:
    stem = p.stem.removesuffix("_skeleton")
    payload = torch.load(p, map_location="cpu", weights_only=False)
    diag_log = payload.get("diagnostics")
    skeletons[stem] = dict(
        payload=payload, diag_df=pd.DataFrame(diag_log) if diag_log else None,
        label=LABELS.get(stem, stem), color=COLORS.get(stem, "grey"), ls=LINESTYLES.get(stem, "-"),
    )

available = [s for s in skeletons if skeletons[s]["diag_df"] is not None]
print(f"{DATASET} split_{SPLIT_ID:02d}: {len(skeletons)} skeletons loaded, "
      f"{len(available)} with diagnostics: {[skeletons[s]['label'] for s in available]}")


## Figure 1 — the rate $\lambda(t)$ against its grid bound

Between skeleton points $(x_n,v_n)\to(x_{n+1},v_{n+1})$, the flow is deterministic (ZigZag:
linear; Boomerang: harmonic orbit around $x_{\mathrm{ref}}$), so $\lambda(t)$ is closed-form
on $t\in[0,\Delta t_n]$. ZigZag: $\lambda(t)=\sum_j\mathrm{clamp}(v_j\nabla_jU(x_t),0)+D\gamma
\ge D\gamma$. Boomerang: $\lambda(t)=\langle v_t,\,\nabla U(x_t)-\Sigma^{-1}(x_t-x_{\mathrm
{ref}})\rangle$, signed — thinning clamps it at $\max(\lambda(t),0)$. The grid bound
$\Lambda_i$ is Algorithm 2's piecewise-constant upper envelope over $n_{\mathrm{seg}}$
sub-intervals, plotted directly against $\lambda(t)$ (not summarized) to show the actual
gap the sampler is thinning against. Non-sticky only: sticky's rate also depends on
`frozen_mask`, not saved per skeleton point.


In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    build_zigzag_sampler, build_boomerang_sampler, BNNConfig,
    make_split, load_raw_datasets, build_target, BASE_SEED, DTYPE, DEVICE,
)
from sazz.gpu_friendly.utils.fast_grid_bound import build_grid_bound, build_grid_bound_vectorized

RATE_STEMS = [s for s in ("grid_zigzag", "grid_boomerang") if s in skeletons]
N_INTERVALS, N_T_POINTS, K_STEPS = 6, 10, 10  # K_STEPS: skeleton steps spanned per panel

rate_samplers = {}
for stem in RATE_STEMS:
    payload = skeletons[stem]["payload"]
    cfg = BNNConfig(layer_sizes=payload["layer_sizes"], activation=payload["activation"],
                     prior_sigma_scale=payload["prior_sigma_scale"])
    data = make_split(*load_raw_datasets((DATASET,))[DATASET],
                       seed=BASE_SEED + payload["split_id"], dtype=DTYPE, device=DEVICE)
    bm, x_ref, Sigma_inv = build_target(data, cfg)
    sampler = build_boomerang_sampler(bm, x_ref, Sigma_inv) if stem == "grid_boomerang" \
        else build_zigzag_sampler(bm)
    rate_samplers[stem] = sampler

fig, axes = plt.subplots(len(RATE_STEMS), N_INTERVALS,
                          figsize=(4 * N_INTERVALS, 3.2 * len(RATE_STEMS)), squeeze=False)

for row, stem in enumerate(RATE_STEMS):
    sampler = rate_samplers[stem]
    sk = skeletons[stem]
    payload = sk["payload"]
    positions, velocities, times = payload["positions"], payload["velocities"], payload["times"]
    is_boomerang = stem == "grid_boomerang"

    interval_idx = sorted(set(np.linspace(0, positions.shape[0] - 1 - K_STEPS, N_INTERVALS)
                               .round().astype(int).tolist()))
    for col, n in enumerate(interval_idx):
        ax = axes[row, col]
        x_n, v_n = positions[n], velocities[n]
        dt_n = float(times[n + K_STEPS] - times[n])
        if dt_n <= 0:
            ax.set_title(f"n={n} (dt<=0, skipped)")
            continue

        t_grid = np.linspace(0.0, dt_n, N_T_POINTS)
        raw_rate = np.array([sampler._rate_scalar(float(t), x_n, v_n) for t in t_grid])

        n_segments = int(min(max(math.ceil(dt_n / sampler.grid_spacing), 2), sampler.n_segments))
        rate_and_grad_fn = sampler._make_rate_and_grad_fn(x_n, v_n)
        if is_boomerang:
            knot_t, seg_bounds, _, _ = build_grid_bound(rate_and_grad_fn, dt_n, n_segments,
                                                          sampler.device, sampler.dtype)
        else:
            knot_t, seg_bounds, _, _ = build_grid_bound_vectorized(
                rate_and_grad_fn, dt_n, n_segments, sampler.device, sampler.dtype,
                signed=(sampler.strategy == "vectorized_signed"), offset=sampler.D * sampler.gamma)
        knot_t = knot_t.detach().cpu().numpy()
        seg_bounds_np = np.clip(seg_bounds.detach().cpu().numpy(), 0.0, None)

        if is_boomerang:
            ax.plot(t_grid, raw_rate, color="grey", lw=1.0, ls=":", label=r"signed $\lambda(t)$")
            ax.plot(t_grid, np.clip(raw_rate, 0.0, None), color=sk["color"], lw=1.6,
                     label=r"$\max(\lambda(t), 0)$")
            ax.axhline(0.0, color="black", lw=0.6, alpha=0.5)
        else:
            ax.plot(t_grid, raw_rate, color=sk["color"], lw=1.6, label=r"$\lambda(t) \geq D\gamma$")

        ax.plot(np.append(knot_t[:-1], dt_n), np.append(seg_bounds_np, seg_bounds_np[-1]),
                color="crimson", lw=1.2, drawstyle="steps-post", alpha=0.8, label=r"bound $\Lambda_i$")
        ax.set_title(f"n={n}\N{RIGHTWARDS ARROW}{n + K_STEPS}, " + r"$\Delta t=$" + f"{dt_n:.2e}")
        ax.set_xlabel("$t$")
        if col == 0:
            ax.set_ylabel(f"{sk['label']}\n" + r"$\lambda(t)$")

    for col in range(len(interval_idx), N_INTERVALS):
        axes[row, col].set_visible(False)
    axes[row, len(interval_idx) - 1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))

fig.suptitle(f"{DATASET.capitalize()} — rate vs. grid bound ({K_STEPS} skeleton steps/panel)", y=1.02)
plt.tight_layout()
plt.show()

# --- n_segments summary, computed from every real per-call horizon in the
# run's diagnostics log (not just the 6 plotted panels above) ---
n_segments_rows = []
for stem in RATE_STEMS:
    sampler = rate_samplers[stem]
    diag_df = skeletons[stem]["diag_df"]
    horizons = diag_df["horizon"].to_numpy()
    n_segs = np.clip(np.ceil(horizons / sampler.grid_spacing), 2, sampler.n_segments).astype(int)
    vals, counts = np.unique(n_segs, return_counts=True)
    for v, c in zip(vals, counts):
        n_segments_rows.append({
            "Sampler": skeletons[stem]["label"], "n_segments": int(v),
            "count": int(c), "fraction": c / len(n_segs),
        })

n_segments_df = (pd.DataFrame(n_segments_rows)
                  .pivot(index="n_segments", columns="Sampler", values="fraction")
                  .fillna(0.0)
                  .sort_index())
n_segments_df.style.format(precision=4, na_rep="—").background_gradient(cmap="Blues", axis=0)


## Table — event-type composition

Every sampler-loop iteration ends in exactly one event: **bounce** (thinning accepted),
**no_event** (thinning rejected the whole grid window — pure overhead), **freeze**/**thaw**
(sticky-only), **refresh** (Boomerang-only, full velocity resample). Entries are each
type's share of total loop iterations.


In [ ]:
EVENT_TYPES = ["bounce", "no_event", "freeze", "thaw", "refresh"]

event_rows = []
for stem in available:
    df = skeletons[stem]["diag_df"]
    row = {"Sampler": skeletons[stem]["label"]}
    row.update({e: (df["event_type"] == e).mean() for e in EVENT_TYPES})
    event_rows.append(row)

event_df = pd.DataFrame(event_rows).set_index("Sampler")
event_df.style.format(precision=4, na_rep="—").background_gradient(cmap="Blues", axis=None)


## Figure 2 — thinning-bound tightness

$\mathrm{max\_ratio} = \lambda(\tau)/\Lambda(\tau)$ at the accepted event time $\tau$ (1 =
bound exactly tight, 0 = bound was far above the realized rate). Only defined, and only
plotted, on **bounce** rows — `no_event`/freeze/thaw rows have `max_ratio=0` by construction
(no accept was ever tested against the bound), so including them would just show the
event-type mix from the table above, not bound quality. Left: distribution over all bounces.
Right: rolling mean over loop iterations, showing whether the bound tightens as
`grid_t_max`/`grid_spacing` adapt.


In [ ]:
ROLL_WINDOW = 500

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for stem in available:
    sk = skeletons[stem]
    df = sk["diag_df"]
    bounces = df.loc[df["event_type"] == "bounce", "max_ratio"]

    axes[0].hist(bounces, bins=50, range=(0, 1), density=True, alpha=0.4,
                 color=sk["color"], histtype="stepfilled", label=sk["label"])

    rolling = bounces.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW // 2).mean()
    axes[1].plot(rolling.to_numpy(), color=sk["color"], ls=sk["ls"], lw=1.3,
                 alpha=0.9, label=sk["label"])

axes[0].set(xlabel="max_ratio (bounce events only)", ylabel="Density",
            title=f"{DATASET.capitalize()} — bound tightness")
axes[0].legend(fontsize=8)
axes[1].set(xlabel="Bounce index", ylabel=f"Rolling mean (window={ROLL_WINDOW})",
            ylim=(0, 1), title=f"{DATASET.capitalize()} — bound tightness over the run")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

for stem in available:
    mr = skeletons[stem]["diag_df"].query("event_type == 'bounce'")["max_ratio"]
    print(f"{skeletons[stem]['label']}: max_ratio mean={mr.mean():.4f}  median={mr.median():.4f}")


## Posterior noise from the raw skeleton (grid Boomerang)

Sanity check: resample $\sigma$ from the continuous Boomerang path via
`resample_boomerang_path_torch`, which reconstructs the deterministic flow between skeleton
points using the saved `velocities`.


In [ ]:
from sazz.gpu_friendly.utils.resample import resample_boomerang_path_torch

N_RESAMPLE_CHECK = 50_000
BURNIN_FRAC_CHECK = 0.2

BOOM_STEM = "grid_boomerang"
assert BOOM_STEM in skeletons, f"no {BOOM_STEM} skeleton loaded"

payload = skeletons[BOOM_STEM]["payload"]
x_ref = payload["x_ref"].to(dtype=DTYPE)

samples = resample_boomerang_path_torch(
    payload["positions"].to(dtype=DTYPE), payload["velocities"].to(dtype=DTYPE),
    payload["times"].to(dtype=DTYPE), x_ref,
    N_resample=N_RESAMPLE_CHECK, burnin_frac=BURNIN_FRAC_CHECK,
)
noise_samples = samples[:, -1].exp().numpy()  # sigma, standardised scale

print(f"{skeletons[BOOM_STEM]['label']}: {len(noise_samples)} resampled draws")
print(f"sigma: mean={noise_samples.mean():.4f} median={np.median(noise_samples):.4f} "
      f"std={noise_samples.std():.4f} min={noise_samples.min():.4f} max={noise_samples.max():.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(noise_samples, bins=60, color=skeletons[BOOM_STEM]["color"], alpha=0.6)
ax.axvline(noise_samples.mean(), color="black", ls="--", lw=1, label=f"mean={noise_samples.mean():.3f}")
ax.set(xlabel=r"Posterior $\sigma$ (resampled draws, standardised scale)", ylabel="Count",
       title=f"{DATASET.capitalize()}, {skeletons[BOOM_STEM]['label']} — noise from raw skeleton")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
